OR calculation across the CVFG dataframe

In [1]:
# Make sure our package is installed (run once)
#!pip install -e /home/dataproc/repos/igvf-cvfg-biobank

Notes on running in AoU's DataProc: as of 2026-09-25, trying to use pandas with the `fastparquet` engine throws an error. Manually forcing an update of relevant packages resolves it:

In [2]:
# Run once in dataproc
#!pip install --upgrade numpy numba fastparquet pandas statsmodels

In the standard environment, `fastparquet` is not installed, but we use it to append to a parquet file:

In [3]:
# Run once in a standard env
#!pip install fastparquet

In [4]:
# Common imports
import os
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

Note that we need to use the bucket path for the WGS table in dataproc, but we point to the local mount in the standard environment. Unsure why the former does't work in both (as of 2026-09-25).

In [5]:
# Constants (mostly paths to things)

variant_table_path = 'variant_tables/integrated_variant_effects_filtered_2026-09-29.tsv.gz'
google_project = os.getenv('GOOGLE_PROJECT')
tp_carriers_path = Path('carrier_tables/total-points-carriers_2026-10-01.parquet')
demographics_path = '../cohorts/demo.tsv.gz'
ancestry_pca_path = '../cohorts/ancestry_pca.tsv.gz'
or_estimates_path = 'results/total-points-OR-estimates_2026-10-01.parquet'

# In standard env
#wgs_mt_path = f"/home/jupyter/workspace/All of Us Controlled Tier v8/vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/splitMT/hail.mt"

# In dataproc
wgs_mt_path = 'gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/splitMT/hail.mt'


In [6]:
# Load dataframe
variants_df = pd.read_table(variant_table_path, dtype=str)

In [7]:
variants_df

Now establish the sets of variants to test.

First we inspect the columns that will define variant sets:

In [8]:
variants_df[['condensed_consequence']].value_counts(dropna=False)

In [9]:
variants_df['combined_points'].value_counts(dropna=False)

In [10]:
# Limit to genes with case/control cohorts established
genes_with_cohorts = {
    gene for gene in variants_df['Gene'].unique()
    if Path('gene_cohorts', f'{gene} cohort.parquet').exists()
}

In [11]:
from cvfgaou import notation

variant_groups = {
    ('Combined Points', gene, 'ExCALIBR + REVEL', consequence, classification):
        df[classification_selection & consequence_selection]
    for gene, df in variants_df.groupby('Gene')
    for classification, classification_selection in [
        (f'{notation.GEQ_CHAR} {points:+d}', pd.to_numeric(df['combined_points']) >= points)
        for points in range(1,13)
    ] + [
        (f'{notation.LEQ_CHAR} {-points:+d}', pd.to_numeric(df['combined_points']) <= -points)
        for points in range(1,13)
    ]
    for consequence, consequence_selection in (
        ('All', True),
        ('Missense', df['condensed_consequence'] == 'missense_variant'),
        ('Stop Gained', df['condensed_consequence'] == 'stop_gained')
    )
    if gene in genes_with_cohorts
}

Next, get the sets of carriers for each variant group

In [12]:
# First, make sure the position column is an integer and
# make sure that the chromosomes are strings with the "chr" prefix

variant_groups = {
    group_label: df.assign(
        Chromosome = 'chr' + df['Chrom'],
        pos = pd.to_numeric(df['hg38_start'], downcast='integer')
    )
    for group_label, df in variant_groups.items()
    if not df.empty
}

In [13]:
for label, df in variant_groups.items():
    if df[['Chromosome', 'pos', 'ref_allele', 'alt_allele']].isna().any(axis=None):
        display(label)
        display(df)

In [14]:
import os
import hail as hl

In [15]:
hl.init(
    #default_reference='GRCh38', deprecated
    gcs_requester_pays_configuration=google_project
)

In [16]:
# WGS table

wgs_mt = hl.read_matrix_table(wgs_mt_path)

In [17]:
from cvfgaou import hailtools

In [18]:
# Index column labels
index_cols=['Dataset/Predictor', 'Gene', 'Grouping/Calibration', 'Consequence', 'Group/Range']

# Attempt to check if we are adding to an existing file
excluded_groups = set()
if tp_carriers_path.exists():
    carriers_df = pd.read_parquet(tp_carriers_path)
    excluded_groups = set(
        carriers_df.
        index.
        to_frame()[index_cols].
        drop_duplicates().
        itertuples(index=False, name=None)
    )
    display(f'Existing file found with {len(excluded_groups)} groups.')

for variant_group, df in tqdm(variant_groups.items()):
    if variant_group in excluded_groups:
        continue
    carrier_df = hailtools.get_cols_with_variants(
        df[['Chromosome', 'pos', 'ref_allele', 'alt_allele']], wgs_mt,
        contig_col='Chromosome',
        pos_col='pos',
        ref_col='ref_allele',
        alt_col='alt_allele',
        reference_genome='GRCh38'
    ).set_index('s')
    labeled_carrier_df = pd.concat(
        {variant_group: carrier_df},
        names=index_cols
    )

    labeled_carrier_df.to_parquet(
        tp_carriers_path,
        engine='fastparquet',
        append=tp_carriers_path.exists()
    )

In [20]:
carriers_df = pd.read_parquet(tp_carriers_path)

In [21]:
carriers_df

OR calculation

In [22]:
demographics_df = pd.read_table(demographics_path, index_col='person_id')
demographics_df

In [23]:
pca_df = pd.read_table(ancestry_pca_path, index_col='research_id')
pca_df

In [24]:
from functools import cache

@cache
def assemble_cohort(gene):
    result = pd.read_parquet(f'gene_cohorts/{gene} cohort.parquet').join(demographics_df).join(pca_df)
    # Trichotomize sex
    result.sex_at_birth.where(
        result.sex_at_birth.isin(['Male', 'Female']),
        'Other',
        inplace=True
    )
    
    return result


In [25]:
from cvfgaou import odds

In [ ]:
or_estimates_dict = {
    (dataset, gene, grouping, consequence, group):
        odds.estimate_logOR(
            pd.to_numeric(exposure_df.index.to_frame().s, downcast='integer'),
            assemble_cohort(gene)
        )
    for (dataset, gene, grouping, consequence, group), exposure_df
    in tqdm(carriers_df.groupby(index_cols))
}

In [ ]:
or_estimates_df = pd.DataFrame.from_dict(or_estimates_dict, orient='index')
or_estimates_df.index.set_names(index_cols, inplace=True)
or_estimates_df

In [ ]:
or_estimates_df.to_parquet(or_estimates_path)